In [1]:
# Import libraries
import pygame
import gymnasium as gym
import numpy as np
import copy
import itertools
import math
np.random.seed(33) # seeding

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# convert binary list to decimal
def binary_list_to_decimal(bin_list):
    bin = ''
    for b in bin_list:
        bin += str(b)
    dec = int(bin,2)
    return dec

# Function to check if a point is inside a polygon (Ray-casting algorithm)
def is_inside_polygon(point, poly):
    x, y = point
    inside = False
    n = len(poly)
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if min(p1y, p2y) < y <= max(p1y, p2y) and x <= max(p1x, p2x):
            if p1y != p2y:
                xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
            if p1x == p2x or x <= xinters:
                inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Function to return minimum distance in a list of points
def min_dist(x):
    x = np.array(x).astype('float32')
    dists = []
    for p1, p2 in itertools.combinations(x, 2):
        dist = np.linalg.norm(p1-p2)
        dists.append(dist)
    return float(np.min(dists))

In [3]:
experiments_path = r'./experiment_sets.txt'
# Read the experiments file and select the experiment
with open(experiments_path, 'r') as experiment_file:
    codes = experiment_file.read()
    exec(codes) # execute
selected_experiment = set3 # Select the experiment set
selected_experiment

{'field': [(13.0, 23.0), (27.0, 21.0), (31.0, 32.0), (27.0, 33.0)],
 'init_positions': [array([20., 25.]), array([30., 30.]), array([25., 30.])],
 'infected_locations': {(16.0, 25.0),
  (20.0, 24.0),
  (21.0, 28.0),
  (24.0, 30.0),
  (27.0, 23.0),
  (27.0, 28.0)}}

In [4]:
sf = 10 # scaling factor
selected_experiment['field'] = [(x*sf, y*sf) for (x,y) in selected_experiment['field']]
selected_experiment['infected_locations'] = [(x*sf, y*sf) for (x,y) in selected_experiment['infected_locations']]
selected_experiment['init_positions'] = [v*sf for v in selected_experiment['init_positions']]
selected_experiment, len(selected_experiment['init_positions']), len(np.unique(selected_experiment['init_positions'], axis=0))

({'field': [(130.0, 230.0), (270.0, 210.0), (310.0, 320.0), (270.0, 330.0)],
  'init_positions': [array([200., 250.]),
   array([300., 300.]),
   array([250., 300.])],
  'infected_locations': [(240.0, 300.0),
   (160.0, 250.0),
   (200.0, 240.0),
   (270.0, 280.0),
   (210.0, 280.0),
   (270.0, 230.0)]},
 3,
 3)

Make the multi-agent class:

In [ ]:
class MultiRobotEnv(gym.Env):
    metadata = {'render_modes': ['human', 'print', 'rgb_array'], "render_fps": 4}
    def __init__(self, render_mode=None, field_info=copy.deepcopy(selected_experiment), wind_par=[0,0], num_robots=3):
        super(MultiRobotEnv, self).__init__()
        # Screen dimensions
        self.edge_buffer = 10 # Boundary above the max values
        self.poly_vertices = field_info['field'] # Vertices of polygon
        self.xs, self.ys = zip(*field_info['field']) # x and y values of the vertices of the polygonal field
        self.WIDTH, self.HEIGHT = 1000, 1000 # Use this if we want to have fixed width and height  
        # self.WIDTH, self.HEIGHT = max(self.xs) + self.edge_buffer, max(self.ys) + self.edge_buffer        

        # Robot parameters
        self.num_robots = num_robots # Rendering error if more than 7
        self.init_robot_positions = np.array(field_info['init_positions'])[:self.num_robots]
        self.robot_size = 10
        self.mass = 1.0
        self.thrust_power = 0.5  # Force applied per action
        self.max_speed = 5  # Maximum speed    
        self.min_speed = -5 # Minimum speed
        self.min_positions = np.zeros(self.num_robots*2) # Minimum positions
        self.max_positions = np.array([[self.WIDTH, self.HEIGHT] for _ in range(self.num_robots)]) # Maximum positions
        self.min_velocities = np.array([[self.min_speed, self.min_speed] for _ in range(self.num_robots)]) # Min speed list
        self.max_velocities = np.array([[self.max_speed, self.max_speed] for _ in range(self.num_robots)]) # Max speed list
        self.wind_f_a, self.wind_beta_a = wind_par # Wind parameters: magnitude and angle

        # infected locations
        self.initial_inf_locations = field_info['infected_locations']
        self.infected_size = 10 # Radius of infected locations
        self.infected_length = len(field_info['infected_locations'])
        self.infected_state_length = 2**(self.infected_length) # 2**5, binary to decimal

        # Action space: thrust in x and y directions for each robot
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(self.num_robots, 2), dtype=np.float32)

        # Observation space: position and velocity (x, y, vx, vy) for each robot + infected location        
        self.observation_space = gym.spaces.Box( # The (visited) weed locations are tracked on the observation space
                    low = np.concatenate((self.min_positions.flatten(), self.min_velocities.flatten(), np.array([0]))), # Lowest positions and velocities
                    high = np.concatenate((self.max_positions.flatten(), self.max_velocities.flatten(), np.array([self.infected_state_length - 1]))), # highest positions and velocities
                    dtype=np.float32)

        assert render_mode is None or render_mode in self.metadata["render_modes"] # Check if the render mode is correct
        self.render_mode = render_mode
        self.screen = None
        self.clock = None
        # If human-rendering is used, `self.screen` will be a reference to the screen that we draw to. `self.clock` will be a clock that is used
        # to ensure that the environment is rendered at the correct framerate in human-mode. They will remain `None` until human-mode is used for the first time.   

        # Reset the environment and start
        self.reset()
    
    def _get_obs(self):
        info = {f'robot{i}': self.robot_positions[i] for i in range(self.num_robots)} # Current position of each robot
        infected = binary_list_to_decimal(list(self.infected_dict.values())) # Convert the binary list of infected locations to a decimal value
        state = np.concatenate((self.robot_positions.flatten(), self.robot_velocities.flatten(), np.array([infected])), dtype=np.float32) # Current state of the robots
        return state, info        

    def reset(self, seed=None, options={}):
        # Reset the visited states and counts
        self.step_count = 0
        self.visited = set()
        self.infected_locations = copy.deepcopy(self.initial_inf_locations) # Initial infected locations
        self.infected_dict = {v:0 for v in self.infected_locations} # 0 for unvisited infected locations, 1 for visited
        self.robot_positions = copy.deepcopy(self.init_robot_positions) # Initial positions of each robot
        self.robot_velocities = np.zeros((self.num_robots, 2)) # Initial velocities of each robot (zero)
        return self._get_obs()
    
    def step(self, actions):
        terminated, truncated = False, False
        rewards = 0
        self.step_count += 1
        for i in range(self.num_robots): # For every robot
            ax, ay = actions[i] * self.thrust_power # What actions to take

            # Update velocity
            self.robot_velocities[i][0] += ax / self.mass + self.wind_f_a * np.cos(np.radians(self.wind_beta_a))
            self.robot_velocities[i][1] += ay / self.mass + self.wind_f_a * np.sin(np.radians(self.wind_beta_a))

            # Limit velocity
            self.robot_velocities[i] = np.clip(self.robot_velocities[i], self.min_speed, self.max_speed)

            # Predict new position
            new_position = self.robot_positions[i] + self.robot_velocities[i]

            # Boundary conditions (keep robot within polygon)
            if is_inside_polygon(new_position, self.poly_vertices):
                pass
            else: # Hits the wall!
                rewards -= 10000 # Medium negative reward for hitting the wall
                self.robot_velocities[i][:] = 0 # Stop movement

            # Update position
            self.robot_positions[i] += self.robot_velocities[i]
            
            # Boundary conditions (keep robot within screen)
            self.robot_positions[i] = np.clip(self.robot_positions[i], [0, 0], [self.WIDTH, self.HEIGHT])

            # Check if location is visited before, and add it to the visited locations
            if tuple(self.robot_positions[i]) in self.visited:
                rewards -= 100 # Small negative reward for visiting previous location
            else:
                rewards -= 10 # Very small negative reward for visiting new locations
            self.visited.add(tuple(self.robot_positions[i]))            

            # Check if any infected location is visited        
            nearby_infected_locations = [] # To store the nearby infected locations
            for j, inf_loc in enumerate(self.infected_locations): # Loop through each infected location
                dist = np.linalg.norm(self.robot_positions[i]-inf_loc) # Distance between robot position and infected location
                if dist <= self.infected_size: # If the distance is within the radius of the infected location size
                    nearby_infected_locations.append(inf_loc) # Add the infected location
                    rewards += 10000 # Medium positive rewards for visiting each infected location
                    # input("Pause!") # Only pause if you want to visualize visiting infected locations
            for inf_loc in nearby_infected_locations:
                self.infected_locations.remove(inf_loc) # Delete each visited infected location
                self.infected_dict[tuple(inf_loc)] = 1 # Update the infected dictionary
        
        # Check if all infected locations are visited
        if len(self.infected_locations) == 0:
            rewards += 100000 # Big positive rewards for visiting all infected locations
            terminated = True
        
        # Check if any collisions occurred
        if self.num_robots > 1:
            min_dist_between_robots = min_dist(self.robot_positions) # Minimum distance between robots
            if min_dist_between_robots < self.robot_size:
                rewards = -100000 # Big negative rewards for collisions
                terminated = True

        obs, info = self._get_obs() # Get the updated observations
        # rewards = rewards * self.gamma ** self.step_count
        return obs, rewards, terminated, truncated, info
    
    def render(self):
        # Initialize pygame
        if self.screen is None and self.render_mode == "human": # Initialize pygame if it is not initialized
            pygame.init()
            pygame.display.init()
            self.screen = pygame.display.set_mode((self.WIDTH, self.HEIGHT))
            pygame.display.set_caption("Multi-robot RL Environment")
            if self.clock is None:
                self.clock = pygame.time.Clock()
                self.running = True
        
        self.screen.fill((255, 255, 255)) # White color for the background
        colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 128, 0), (128, 0, 255), (255, 0, 255), (128, 128, 128)]  # Colors for each robot: Red, Green, Blue, Orange, Violet, Pink, Grey
        pix_size = 10

        # Draw the polygon
        # pixel_poly_vertices = [(point[0] * pix_size, point[1] * pix_size) for point in self.poly_vertices]
        pygame.draw.polygon(surface=self.screen, 
                            color=(255, 255, 0), # Yello color for the polygon
                            points=self.poly_vertices)
        
        # Draw the visited regions
        for point in self.visited:
            pygame.draw.circle(self.screen, pygame.Color(100, 100, 100, a=0.2), point, pix_size/2) # Light grey color for visited regions, with transparency alpha

        # Draw robots
        for i in range(self.num_robots):
            pygame.draw.circle(self.screen, colors[i], (int(self.robot_positions[i][0]), int(self.robot_positions[i][1])), pix_size/2) # Pick the colors from above list

        # Draw infected locations
            for l in self.infected_locations:
                pygame.draw.circle(self.screen, (0, 255, 255), (int(l[0]), int(l[1])), pix_size/2) # Cyan color for infected locations
        
        pygame.display.flip() # Allows only a portion of the screen to be updated
        self.clock.tick(60)
    
    def close(self):
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()

In [6]:
# Register environment
gym.register(id='MultiRobotEnv-v0', 
             entry_point=MultiRobotEnv,
             max_episode_steps=1000)

# Inference

Load trained network:

In [8]:
from sb3_contrib import CrossQ

weights_path = rf"C:\Users\choton\OneDrive - Kansas State University\PhD Projects\Reinforcement Learning\Codes\for_coRL\github\FlowBotic\trained_models\new_mar25_env3_CrossQ.zip"
# weights_path = rf"C:\Users\choton\OneDrive - Kansas State University\PhD Projects\Reinforcement Learning\Codes\for_coRL\from_gpu_server\models\new_apr11_env1_double_CrossQ.zip"

# Load trained network
model = CrossQ.load(weights_path)
assert False, "Open the scene file in CoppeliaSim before running the below cells"

AssertionError: Open the scene file in CoppeliaSim before running the below cells

## Simulation

**IMPORTANT**: open the scene file using CoppeliaSim robot simulator before running the below cells

In [9]:
from coppeliasim_zmqremoteapi_client import RemoteAPIClient
import numpy as np

# Start the remote API client
client = RemoteAPIClient()
sim = client.getObject('sim')
defaultIdleFps = sim.getInt32Param(sim.intparam_idle_fps)
sim.setInt32Param(sim.intparam_idle_fps, 0)

class Drone_simulator:    
    def __init__(self, polygon, scaling_factor, height, num_robots=3):
        self.scaling_factor = scaling_factor
        self.scaled_polygon = [(x/scaling_factor,y/scaling_factor) for (x,y) in polygon]
        self.rounded_polygon = self.scaled_polygon + [self.scaled_polygon[0]]
        self.color = [[255,0,0],[255,0,255],[0,0,255]]
        self.edges_3d = self.calc_edges_3d()
        self.height = height
        self.num_robots = num_robots

    def start_simulation(self):
        self.trace_line = sim.addDrawingObject(sim.drawing_lines, 5, 0, -1, 9999, [255,0,0]) # red line
        sim.startSimulation()
        print('Program started')

    def stop_simulation(self):
        sim.removeDrawingObject(self.trace_line)
        sim.stopSimulation()

    def calc_edges_3d(self):  # To calculate the edges in the polygon
        edges = []
        for i in range(len(self.rounded_polygon) - 1):
            edges.append([list(self.rounded_polygon[i]), list(self.rounded_polygon[i+1])])
        return edges

    def draw_field(self):
        white = [255, 255, 255]
        lineContainer = sim.addDrawingObject(sim.drawing_lines, 5, 0, -1, 9999, white)
        for l in self.edges_3d: # Drawing the field with white lines
            line = l[0] + [self.height] + l[1] + [self.height]
            for j in range(len(line)):
                if line[j] != self.height:
                    line[j] = int(line[j])
            # print(line)
            sim.addDrawingObjectItem(lineContainer, line)

    def set_agent_positions(self, info):
        for i in range(self.num_robots):
            drone = '/Quadcopter['
            obj_path = drone+str(i)+']'
            objHandle = sim.getObject(obj_path)
            print(np.append(info['robot'+str(i)],[self.height]))
            x = info['robot'+str(i)]
            x = [xi/self.scaling_factor for xi in x]
            x = x + [self.height]
            print(x)
            sim.setObjectPosition(objHandle, -1, x) # Initiate the position of the robots
    
    def set_weed_locations(self, weed_locations):
        weed_obj = sim.getObject('/weed')
        for i, loc in enumerate(weed_locations):
            new_weed_obj = sim.copyPasteObjects([weed_obj])[0]
            x = [xi/self.scaling_factor for xi in loc]
            new_pos = x + [0]
            sim.setObjectPosition(new_weed_obj, -1, new_pos)

    def move_agents(self, info):
        for i in range(self.num_robots):
            obj_path = '/target[' + str(i) + ']'
            objHandle = sim.getObject(obj_path)
            prev_pos = sim.getObjectPosition(objHandle, -1) # current object position
            print(np.append(info['robot'+str(i)],[self.height]))
            x = info['robot'+str(i)] # Get the x,y from info of gym env
            x = [xi/self.scaling_factor for xi in x] # scale the x,y
            x = x + [self.height] # add the z (height)
            # print(x)
            sim.setObjectPosition(objHandle, -1, x) # Initiate the position of the robots
            # draw the line
            line_data = prev_pos + x
            sim.addDrawingObjectItem(self.trace_line, line_data)

Simulation using trained network

In [16]:
# Height of movement
height = 0.35

# Make the environment
env = gym.make('MultiRobotEnv-v0', render_mode='human')
env.metadata['render_fps'] = 1
obs, info = env.reset()
env.render()

# Make the simulator object, draw the field, and set agent positions
drone_simulator = Drone_simulator(polygon=env.unwrapped.poly_vertices, scaling_factor=50, height=height)
drone_simulator.draw_field()
drone_simulator.set_agent_positions(info=info)
drone_simulator.set_weed_locations(weed_locations=env.unwrapped.initial_inf_locations)

[200.   250.     0.35]
[np.float64(4.0), np.float64(5.0), 0.35]
[300.   300.     0.35]
[np.float64(6.0), np.float64(6.0), 0.35]
[250.   300.     0.35]
[np.float64(5.0), np.float64(6.0), 0.35]


In [17]:
# Start simulation
drone_simulator.start_simulation()
terminated, truncated = False, False
total_rewards = 0
while True:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated,  info = env.step(list(action))
    env.render()
    total_rewards += reward
    print(f"Obs: {obs}, Reward: {reward}, terminated: {terminated}, total_rewards: {total_rewards}, action: {action}")
    if terminated or truncated:
        print('terminated:', terminated, 'truncated:', truncated)
        break
    pygame.event.get()
    drone_simulator.move_agents(info=info) # Simulate


Program started
Obs: [ 2.0049568e+02  2.4952361e+02  2.9984314e+02  3.0003940e+02
  2.4950555e+02  3.0039648e+02  4.9567699e-01 -4.7638831e-01
 -1.5684676e-01  3.9407313e-02 -4.9444377e-01  3.9649332e-01
  4.0000000e+01], Reward: 19970, terminated: False, total_rewards: 19970, action: [[ 0.991354   -0.9527766 ]
 [-0.31369352  0.07881463]
 [-0.98888755  0.79298663]]
[200.49567699 249.52361169   0.35      ]
[299.84315324 300.03940731   0.35      ]
[249.50555623 300.39649332   0.35      ]
Obs: [ 2.0145691e+02  2.4858844e+02  2.9920911e+02  2.9965997e+02
  2.4851170e+02  3.0030487e+02  9.6122849e-01 -9.3517751e-01
 -6.3403422e-01 -3.7943035e-01 -9.9385917e-01 -9.1617316e-02
  4.0000000e+01], Reward: -30, terminated: False, total_rewards: 19940, action: [[ 0.931103   -0.9175784 ]
 [-0.9543749  -0.83767533]
 [-0.99883074 -0.97622126]]
[201.45690548 248.58843419   0.35      ]
[299.20911902 299.65997696   0.35      ]
[248.51169708 300.304876     0.35      ]
Obs: [ 2.0291469e+02  2.4721033e+02 

In [18]:
drone_simulator.stop_simulation()
env.close()